# PayIntel AI — Evaluation
Synthetic gold labels; metrics computed rather than assumed.


In [ ]:
from pathlib import Path
import csv
from rfp_analyzer import extract, analyze
root=Path("data")
rfps={p.name:p.read_text() for p in (root/"rfps").glob("*.txt")}
portfolio={p.name:p.read_text() for p in (root/"portfolio").glob("*.txt")}
gold={(r["rfp"],r["capability"]) for r in csv.DictReader(open(root/"gold_labels.csv"))}
pred={(name,cap) for name,txt in rfps.items() for cap in extract(txt)}
tp=len(gold & pred);fp=len(pred-gold);fn=len(gold-pred)
precision=tp/max(1,tp+fp);recall=tp/max(1,tp+fn)
accuracy=tp/max(1,tp+fp+fn) # positive-label agreement, not TN-inclusive accuracy
rows=analyze(rfps,portfolio)
gold_counts={c:sum(1 for name,cap in gold if cap==c) for c in {cap for _,cap in gold}}
count_accuracy=sum(r["RFP Count"]==gold_counts[r["Capability"]] for r in rows)/max(1,len(rows))
supported=set().union(*(extract(txt) for txt in portfolio.values()))
gold_gaps={c for c in gold_counts if c not in supported}
pred_gaps={r["Capability"] for r in rows if r["Status"]=="Missing"}
gap_accuracy=len(gold_gaps & pred_gaps)/max(1,len(gold_gaps|pred_gaps))
false_gap_rate=len(pred_gaps-gold_gaps)/max(1,len(pred_gaps))
citation_accuracy=sum(all(f in rfps for f in r["RFP Sources"].split(", ")) for r in rows)/max(1,len(rows))
print({"requirement_positive_label_accuracy":accuracy,"precision":precision,"recall":recall,"demand_count_accuracy":count_accuracy,"gap_identification_jaccard":gap_accuracy,"citation_filename_validity":citation_accuracy,"false_gap_rate":false_gap_rate})
